# Feast Operator with RBAC Configuration
## Objective

This demo provides a reference implementation of a runbook on how to enable Role-Based Access Control (RBAC) for Feast using the Feast Operator with the Kubernetes authentication type.

The demo steps include deploying the Feast Operator, creating Feast instances with server components (registry, offline store, online store), and client examples within a Kubernetes environment. The goal is to ensure secure access control for Feast instances deployed by the Feast Operator.
 
Please read these reference documents for understanding the Feast RBAC framework.
- [RBAC Architecture](https://docs.feast.dev/v/master/getting-started/architecture/rbac) 
- [RBAC Permission](https://docs.feast.dev/v/master/getting-started/concepts/permission).
- [RBAC Authorization Manager](https://docs.feast.dev/v/master/getting-started/components/authz_manager)


## Prerequisites
* Kubernetes Cluster
* [kubectl](https://kubernetes.io/docs/tasks/tools/#kubectl) Kubernetes CLI tool.

## Install Prerequisites

The following commands install and configure all the prerequisites on a MacOS environment. You can find the
equivalent instructions on the offical documentation pages:
* Install the `kubectl` cli.
* Install Kubernetes and Container runtime (e.g. [Colima](https://github.com/abiosoft/colima)).
  * Alternatively, authenticate to an existing Kubernetes or OpenShift cluster.
  
```bash
brew install colima kubectl
colima start -r containerd -k -m 3 -d 100 -c 2 --cpu-type max -a x86_64
colima list
```

In [29]:
!kubectl create ns feast
!kubectl config set-context --current --namespace feast

namespace/feast created
Context "kind-kind" modified.


Validate the cluster setup:

In [30]:
!kubectl get ns feast

NAME    STATUS   AGE
feast   Active   8s


## Install the Feast Operator

In [31]:
## Use this install command from a release branch (e.g. 'v0.43-branch')
!kubectl apply -f ../../infra/feast-operator/dist/install.yaml

## OR, for the latest code/builds, use one the following commands from the 'master' branch
# !make -C ../../infra/feast-operator install deploy IMG=quay.io/feastdev-ci/feast-operator:develop FS_IMG=quay.io/feastdev-ci/feature-server:develop
# !make -C ../../infra/feast-operator install deploy IMG=quay.io/feastdev-ci/feast-operator:$(git rev-parse HEAD) FS_IMG=quay.io/feastdev-ci/feature-server:$(git rev-parse HEAD)

!kubectl wait --for=condition=available --timeout=5m deployment/feast-operator-controller-manager -n feast-operator-system

namespace/feast-operator-system created
customresourcedefinition.apiextensions.k8s.io/featurestores.feast.dev created
serviceaccount/feast-operator-controller-manager created
role.rbac.authorization.k8s.io/feast-operator-leader-election-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-featurestore-editor-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-featurestore-viewer-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-manager-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-metrics-auth-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-metrics-reader created
rolebinding.rbac.authorization.k8s.io/feast-operator-leader-election-rolebinding created
clusterrolebinding.rbac.authorization.k8s.io/feast-operator-manager-rolebinding created
clusterrolebinding.rbac.authorization.k8s.io/feast-operator-metrics-auth-rolebinding created
service/feast-operator-controller-manager-metrics-service created
deployment.ap

## Deployment Architecture
The primary objective of this runbook is to guide the deployment of Feast services with RBAC

In this notebook, we will deploy a distributed topology of Feast services, which includes:

* `Registry Server`: Handles metadata storage for feature definitions.
* `Online Store Server`: Uses the `Registry Server` to query metadata and is responsible for low-latency serving of features.
* `Offline Store Server`: Uses the `Registry Server` to query metadata and provides access to batch data for historical feature retrieval.
* `Kubernetes` Authentication types for RBAC Configuration for Feast resources.
* Setting update client RBAC examples to test Feast RBAC based on roles assigned.


## Install the Feast services via FeatureStore CR
Next, we'll use the running Feast Operator to install the feast services with Server components online, offline, registry with kubernetes Authorization set. Apply the included [reference deployment](../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml) to install and configure Feast with kubernetes Authorization .

In [32]:
!cat ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml
!kubectl apply -f ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml -n feast

apiVersion: feast.dev/v1alpha1
kind: FeatureStore
metadata:
  name: sample-kubernetes-auth
spec:
  feastProject: feast_rbac
  authz:
    kubernetes:
      roles:
        - feast-writer
        - feast-reader
  services:
    offlineStore:
      server: {}
    onlineStore:
      server: {}
    registry:
      local:
        server: {}
    ui: {}
featurestore.feast.dev/sample-kubernetes-auth created


## Validate the running FeatureStore deployment
Validate the deployment status.

In [34]:
!kubectl get all
!kubectl wait --for=condition=available --timeout=8m deployment/feast-sample-kubernetes-auth

NAME                                                READY   STATUS    RESTARTS   AGE
pod/feast-sample-kubernetes-auth-774f6df8df-pcqxz   4/4     Running   0          74s

NAME                                            TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)   AGE
service/feast-sample-kubernetes-auth-offline    ClusterIP   10.96.99.64     <none>        80/TCP    74s
service/feast-sample-kubernetes-auth-online     ClusterIP   10.96.160.127   <none>        80/TCP    74s
service/feast-sample-kubernetes-auth-registry   ClusterIP   10.96.114.53    <none>        80/TCP    74s
service/feast-sample-kubernetes-auth-ui         ClusterIP   10.96.105.73    <none>        80/TCP    74s

NAME                                           READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/feast-sample-kubernetes-auth   1/1     1            1           74s

NAME                                                      DESIRED   CURRENT   READY   AGE
replicaset.apps/feast-sample-kubernetes-auth-774f

Validate that the FeatureStore CR is in a `Ready` state.

In [35]:
!kubectl get feast

NAME                     STATUS   AGE
sample-kubernetes-auth   Ready    81s


## Configure the RBAC Permissions
we have defined permission in `permissions_apply.py`.

In [36]:
#view the permissions  
!cat  permissions_apply.py

from feast.feast_object import ALL_RESOURCE_TYPES
from feast.permissions.action import READ, AuthzedAction, ALL_ACTIONS
from feast.permissions.permission import Permission
from feast.permissions.policy import RoleBasedPolicy

admin_roles = ["feast-writer"]
user_roles = ["feast-reader"]

user_perm = Permission(
    name="feast_user_permission",
    types=ALL_RESOURCE_TYPES,
    policy=RoleBasedPolicy(roles=user_roles),
    actions=[AuthzedAction.DESCRIBE] + READ
)

admin_perm = Permission(
    name="feast_admin_permission",
    types=ALL_RESOURCE_TYPES,
    policy=RoleBasedPolicy(roles=admin_roles),
    actions=ALL_ACTIONS
)


In [37]:
# Copy the Permissions to the pods under feature_repo directory
!kubectl cp permissions_apply.py $(kubectl get pods -l 'feast.dev/name=sample-kubernetes-auth' -ojsonpath="{.items[*].metadata.name}"):/feast-data/feast_rbac/feature_repo -c online

In [38]:
#view the feature_store.yaml configuration 
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- cat feature_store.yaml

project: feast_rbac
provider: local
offline_store:
    type: dask
online_store:
    path: /feast-data/online_store.db
    type: sqlite
registry:
    path: /feast-data/registry.db
    registry_type: file
auth:
    type: kubernetes
entity_key_serialization_version: 3


## Apply the Permissions and Feast Object to Registry

In [39]:
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast apply

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_enabled" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_len" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/feast-data/feast_rbac/feature_repo/example_repo.py:27: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'driver'.
  driver = Entity(

**List the applied permission details permissions on Feast Resources.**

In [40]:
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions list-roles
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions list
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions describe feast_admin_permission
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions describe feast_user_permission

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_enabled" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_len" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
+--------------+
| ROLE NAME    |
+==============+
| feast-reader |
+--------------+
| feast-writer |
+--------------+
<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>:

## Client 

**The Operator creates the client ConfigMap containing the feature_store.yaml. We can retrieve it and store it in the feature_repo.**

In [43]:
!kubectl get configmap feast-sample-kubernetes-auth-client -n feast -o jsonpath='{.data.feature_store\.yaml}' > ./client/feature_store.yaml
!cat client/feature_store.yaml

project: feast_rbac
provider: local
offline_store:
    host: feast-sample-kubernetes-auth-offline.feast.svc.cluster.local
    type: remote
    port: 80
online_store:
    path: http://feast-sample-kubernetes-auth-online.feast.svc.cluster.local:80
    type: remote
registry:
    path: feast-sample-kubernetes-auth-registry.feast.svc.cluster.local:80
    registry_type: remote
auth:
    type: kubernetes
entity_key_serialization_version: 3


## Feast RBAC Kubernetes Authentication
This Feast **Role-Based Access Control (RBAC)** in Kubernetes support authentication  **inside a Kubernetes pod** and **outside a pod** when running a local script.
### Inside a Kubernetes Pod
Feast automatically retrieves the Kubernetes ServiceAccount token from:
```
/var/run/secrets/kubernetes.io/serviceaccount/token
```
This means:
- No manual configuration is needed inside a pod.
- The token is mounted automatically and used for authentication.
- Developer just need create the binding with role and service account accordingly.
- Code Reference:  
[Feast Kubernetes Auth Client Manager (Pod Token Usage)](https://github.com/feast-dev/feast/blob/master/sdk/python/feast/permissions/client/kubernetes_auth_client_manager.py#L15) 
- Example 

### Outside a Kubernetes Pod (Local Machine)
If running Feast outside of Kubernetes, authentication requires setting the token manually:
```sh
export LOCAL_K8S_TOKEN="your-service-account-token"
```
Feast will use this token for authentication.

Reference:  
[Feast Authentication via `LOCAL_K8S_TOKEN`](https://github.com/feast-dev/feast/blob/master/sdk/python/feast/permissions/client/kubernetes_auth_client_manager.py#L50)

## Setting Up and Testing Feast Users
The steps below will:
- Create **three different ServiceAccounts** for Feast.
- Assign appropriate **RoleBindings** for access control.
- Retrieve and use the **ServiceAccount token** for authentication.
- Run `test.py` to validate authentication.

Run Port Forwarding for All Services for local testing 

In [59]:
import subprocess

# Define services and their local ports
services = {
    "offline_store": ("feast-sample-kubernetes-auth-offline", 8081),
    "online_store": ("feast-sample-kubernetes-auth-online", 8082),
    "registry": ("feast-sample-kubernetes-auth-registry", 8083),
}

# Start port-forwarding for each service
port_forward_processes = {}
for name, (service, local_port) in services.items():
    cmd = f"kubectl port-forward svc/{service} -n feast {local_port}:80"
    process = subprocess.Popen(cmd, shell=True)
    port_forward_processes[name] = process
    print(f"Port forwarding {service} -> localhost:{local_port}")

Port forwarding feast-sample-kubernetes-auth-offline -> localhost:8081
Port forwarding feast-sample-kubernetes-auth-online -> localhost:8082
Port forwarding feast-sample-kubernetes-auth-registry -> localhost:8083


### Setup & Test Read-Only Feast User (serviceaccount: feast-user-sa, role: feast-reader)

**Step 1: Create the ServiceAccount and Role Binding**

In [ ]:
# Step 1: Create the ServiceAccount
!echo "Creating ServiceAccount: feast-user-sa"
!kubectl create serviceaccount feast-user-sa -n feast

# Step 2: Assign RoleBinding (Read-Only Access for Feast)
!echo "Assigning Read-Only RoleBinding: feast-user-rolebinding"
!kubectl create rolebinding feast-user-rolebinding --role=feast-reader --serviceaccount=feast:feast-user-sa -n feast

In [60]:
import subprocess
import os

def get_k8s_token(service_account):
    namespace = "feast"

    if not service_account:
        raise ValueError("Service account name is required.")

    result = subprocess.run(
        ["kubectl", "create", "token", service_account, "-n", namespace],
        capture_output=True, text=True, check=True
    )

    token = result.stdout.strip()

    if not token:
        return None  # Silently return None if token retrieval fails

    os.environ["LOCAL_K8S_TOKEN"] = token
    return "Token Retrieved: ***** (hidden for security)"


In [ ]:
# Step 2: Assign RoleBinding (Read-Only Access for Feast)

In [61]:
get_k8s_token("feast-user-sa")

'Token Retrieved: ***** (hidden for security)'

In [63]:
!echo "Running Feast RBAC test for Read only User..."
!python client/test.py

Running Feast RBAC test for Read only User...
/Users/ahameed/projects/ai/feast/sdk/python/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
Handling connection for 8083

--- Historical features for training ---
Handling connection for 8081
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.525955
1       1002  ...           20.749933
2       1003  ...           30.925722

[3 rows x 10 columns]

--- Historical features for batch scoring ---
Handling connection for 8081
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.262601
1       1002  ...           20.062418
2       1003  ...           30.311689

[3 rows x 10 columns]

--- Load features into online store/materialize_incremental ---
Materializing 2 feature views to 2025-02-27 11:16:44-05:00 into the remote online store.

Handling connection for 8081
driver_hourly_stats fr

## Setup & Test Admin Feast User (serviceaccount: feast-admin-sa, role: feast-writer)

In [64]:
# Create the ServiceAccount
!echo "Creating ServiceAccount: feast-admin-sa"
!kubectl create serviceaccount feast-admin-sa -n feast

# Assign RoleBinding (Admin Access for Feast)
!echo "Assigning Admin RoleBinding: feast-admin-rolebinding"
!kubectl create rolebinding feast-admin-rolebinding --role=feast-writer --serviceaccount=feast:feast-admin-sa -n feast

# Retrieve and store the token
get_k8s_token("feast-admin-sa")

# Run Feast RBAC test
!echo "Running Feast RBAC test for Admin user (feast-admin-sa)..."
!python client/test.py

Creating ServiceAccount: feast-admin-sa
serviceaccount/feast-admin-sa created
Assigning Admin RoleBinding: feast-admin-rolebinding
rolebinding.rbac.authorization.k8s.io/feast-admin-rolebinding created
Running Feast RBAC test for Admin user (feast-admin-sa)...
/Users/ahameed/projects/ai/feast/sdk/python/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
Handling connection for 8083

--- Historical features for training ---
Handling connection for 8081
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.525955
1       1002  ...           20.749933
2       1003  ...           30.925722

[3 rows x 10 columns]

--- Historical features for batch scoring ---
Handling connection for 8081
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.262601
1       1002  ...           20.062418
2       1003  ...           30.311689

[3 rows x 10 c

### Setup & Test Unauthorized Feast User (serviceaccount: feast-unauthorized-user-sa, role: None)

In [66]:
# Create the ServiceAccount (Without RoleBinding)
!echo "Creating Unauthorized ServiceAccount: feast-unauthorized-user-sa"
!kubectl create serviceaccount feast-unauthorized-user-sa -n feast

# Retrieve and store the token
get_k8s_token("feast-unauthorized-user-sa")

# Run Feast RBAC test (Should fail due to no role assigned)
!echo "Running Feast RBAC test for Unauthorized user (feast-unauthorized-user-sa) - Expected to fail..."
!python client/test.py

Creating Unauthorized ServiceAccount: feast-unauthorized-user-sa
error: failed to create serviceaccount: serviceaccounts "feast-unauthorized-user-sa" already exists
Running Feast RBAC test for Unauthorized user (feast-unauthorized-user-sa) - Expected to fail...
/Users/ahameed/projects/ai/feast/sdk/python/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
Handling connection for 8083

--- Historical features for training ---
An error occurred while fetching historical features: Permission error:
Permission feast_user_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-reader'],Permission feast_admin_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-writer']

--- Historical features for batch scoring ---
An error occurred while fetching historical features: Per

## Summary
| User Type       | ServiceAccount               | RoleBinding Assigned | Expected Behavior in `test.py` |
|----------------|-----------------------------|----------------------|--------------------------------|
| **Read-Only**  | `feast-user-sa`              | `feast-reader`       | Can **read** from the feature store, but **cannot write**. |
| **Admin**      | `feast-admin-sa`             | `feast-writer`       | Can **read and write** feature store data. |
| **Unauthorized** | `feast-unauthorized-user-sa` | _None_               | **Access should be denied** in `test.py`. |

## Uninstall the Operator and all related objects

## Uninstall the Operator and all Feast related objects##

In [23]:
!kubectl delete -f ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml
!kubectl delete -f ../../infra/feast-operator/dist/install.yaml

featurestore.feast.dev "sample-kubernetes-auth" deleted
namespace "feast-operator-system" deleted
customresourcedefinition.apiextensions.k8s.io "featurestores.feast.dev" deleted
serviceaccount "feast-operator-controller-manager" deleted
role.rbac.authorization.k8s.io "feast-operator-leader-election-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-featurestore-editor-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-featurestore-viewer-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-manager-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-metrics-auth-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-metrics-reader" deleted
rolebinding.rbac.authorization.k8s.io "feast-operator-leader-election-rolebinding" deleted
clusterrolebinding.rbac.authorization.k8s.io "feast-operator-manager-rolebinding" deleted
clusterrolebinding.rbac.authorization.k8s.io "feast-operator-metrics-auth-rolebinding" deleted

## Uninstall Client Related Objects

In [24]:
!echo "Deleting RoleBindings..."
!kubectl delete rolebinding feast-user-rolebinding -n feast --ignore-not-found
!kubectl delete rolebinding feast-admin-rolebinding -n feast --ignore-not-found

!echo "Deleting ServiceAccounts..."
!kubectl delete serviceaccount feast-user-sa -n feast --ignore-not-found
!kubectl delete serviceaccount feast-admin-sa -n feast --ignore-not-found
!kubectl delete serviceaccount feast-unauthorized-user-sa -n feast --ignore-not-found


Deleting RoleBindings...
rolebinding.rbac.authorization.k8s.io "feast-user-rolebinding" deleted
Deleting ServiceAccounts...
serviceaccount "feast-user-sa" deleted


Ensure everything has been removed, or is in the process of being terminated.

In [114]:
!kubectl get all -n feast


No resources found in feast namespace.
No resources found in feast-client namespace.


In [67]:
for name, process in port_forward_processes.items():
    process.terminate()
    print(f"Stopped port forwarding for {name}")

Stopped port forwarding for offline_store
Stopped port forwarding for online_store
Stopped port forwarding for registry
